In [9]:
import os
from langchain_groq import ChatGroq
from langchain.agents import create_agent  # <--- Updated Import
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

# Importing your custom tools
from mytools.calcuator import calculator
from mytools.list_files import list_files
from mytools.send_email import send_email
from mytools.file_creation import create_file
from mytools.profit_loss_tool import profit_loss_excel_tool

# 1. Setup Tools
tools = [calculator, list_files, send_email, create_file, profit_loss_excel_tool]

# 2. Initialize the LLM
# Note: Ensure you use a standard Groq model ID like 'llama-3.3-70b-versatile'
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
)

# 3. Setup Memory (Checkpointer)
# This replaces the broken langchain_core.memory logic
memory = InMemorySaver()

# 4. System Prompt
system_instructions = (
    "You are a versatile assistant. You have two ways to respond:\n"
    "1. DIRECT ANSWER: If the user asks a general question, greets you, or asks for an explanation "
    "that does not require external actions, respond directly using your internal knowledge.\n"
    "2. TOOL CALL: Only use a tool if the request explicitly requires it (e.g., calculating math, "
    "sending an email, creating/listing files).\n\n"
    "CRITICAL: Do not use 'create_file' to answer a question. Just type the answer in the chat."
)

# 5. Create the Agent
# We use create_react_agent which is the current standard for tool-calling agents
agent_executor = create_agent(
    llm, 
    tools, 
    checkpointer=memory,
    system_prompt=system_instructions  # <--- Changed from state_modifier to system_prompt
)

# 6. Interactive Loop
# thread_id is the 'key' for the memory. Using the same ID loads previous chat history.
config = {"configurable": {"thread_id": "sana_agent_session_1"}}

print("Agent is ready! (Type 'exit' to quit)")

while True:
    query = input("\nUser: ")
    if query.lower() in ['exit', 'quit']:
        break

    # We send the input as a HumanMessage
    input_state = {"messages": [HumanMessage(content=query)]}
    
    # Streaming the response to see tool calls and the final answer
    for event in agent_executor.stream(input_state, config, stream_mode="values"):
        # The last message in the state is the current AI response
        message = event["messages"][-1]
        message.pretty_print()

Agent is ready! (Type 'exit' to quit)
================================ Human Message =================================

hi
================================== Ai Message ==================================

Hello. It's nice to meet you. Is there something I can help you with or would you like to chat?
================================ Human Message =================================

remember my name is zain
================================== Ai Message ==================================

Nice to meet you, Zain. I'll keep that in mind for our conversation. How's your day going so far?
================================ Human Message =================================

i want to know about wasim akram?
================================== Ai Message ==================================

Wasim Akram is a legendary Pakistani cricketer, widely regarded as one of the greatest fast bowlers in the history of the sport. Born on June 3, 1966, in Lahore, Pakistan, Wasim Akram is known for his exceptional s